<a href="https://colab.research.google.com/github/bahaali273/COMP9170-Hospital-Readmission/blob/bahaa-dev/notebooks/06_gnn_readmission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Graph Neural Network for Hospital Readmission Prediction

**Course:** COMP 9170 — Data Science  
**Author:** Bahaa Al-Juboori

## 1. Research Objective and Graph Construction Strategy

### Objective
Investigate whether a patient-similarity Graph Neural Network
can improve 30-day hospital readmission prediction compared
with conventional tabular machine learning models.

### Proposed Approach
- Represent each patient as a graph node.
- Use clinical features to construct similarity-based edges.
- Build a k-nearest-neighbor (kNN) graph.
- Train a Graph Neural Network for binary node classification.
- Evaluate PR-AUC, ROC-AUC, Precision, Recall, and F1-score.
- Compare the results with Logistic Regression, Random Forest,
  XGBoost, and the regularized MLP.

### Methodological Safeguards
- Use the existing patient-level training and test partitions.
- Fit feature transformations on training data only.
- Construct graph connections without using readmission labels.
- Prevent validation and test labels from entering training.
- Evaluate new patients using a clearly documented graph
  inference strategy.
- Keep the final test set reserved for final evaluation.




## 2. Environment and Dependency Check

### Decisions
- Check Python and PyTorch versions.
- Verify CUDA and GPU availability.
- Check whether PyTorch Geometric is installed.
- Confirm environment compatibility before graph construction.
- Preserve the existing patient-level data partitions.



In [1]:

import sys
import torch
import importlib.util

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        )
    )
else:
    print("GPU: Not detected")

# Check PyTorch Geometric availability
pyg_spec = importlib.util.find_spec("torch_geometric")

print(
    "PyTorch Geometric installed:",
    pyg_spec is not None
)


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56
PyTorch Geometric installed: False



## 3. PyTorch Geometric Installation

### Decisions
- Install PyTorch Geometric for graph neural network experiments.
- Use the existing CUDA-enabled PyTorch environment.
- Verify that GraphSAGE layers are available.
- Test basic GPU graph computation before loading hospital data.


### Results
- Successfully installed PyTorch Geometric version 2.8.1.
- PyTorch version: 2.14.1+cu130.
- CUDA was available.
- Successfully imported PyTorch Geometric Data and SAGEConv.
- Executed a small GraphSAGE compatibility test on the GPU.
- Test output shape: (4, 16).
- Output device: cuda:0.
- A non-blocking torch.jit FutureWarning was displayed.
- The environment is ready for patient-similarity graph experiments.



In [2]:

%pip install -q torch-geometric

import torch
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

print("PyTorch Geometric version:", torch_geometric.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Small GPU compatibility test
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

x = torch.randn(4, 8, device=device)

edge_index = torch.tensor(
    [[0, 1, 2, 3],
     [1, 2, 3, 0]],
    dtype=torch.long,
    device=device
)

layer = SAGEConv(8, 16).to(device)

with torch.no_grad():
    output = layer(x, edge_index)

print("GraphSAGE output shape:", tuple(output.shape))
print("Output device:", output.device)
print("PyG GPU compatibility test passed!")


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


PyTorch Geometric version: 2.8.1
PyTorch version: 2.14.1+cu130
CUDA available: True
GraphSAGE output shape: (4, 16)
Output device: cuda:0
PyG GPU compatibility test passed!



## 4. Dataset Loading and Verification

### Decisions
- Reuse the cleaned hospital dataset from the shared preprocessing pipeline.
- Load the original training and held-out test partitions.
- Verify feature counts, data types, and missing values.
- Preserve the existing patient-level evaluation protocol.
- Avoid using the held-out test set during GNN development.


### Results
- Successfully loaded the cleaned hospital readmission dataset.
- Training dataset: 55,976 rows and 134 columns.
- Held-out test dataset: 13,994 rows and 134 columns.
- Number of input features: 133.
- Training and test feature columns matched exactly.
- Training positive-class prevalence: 8.97%.
- Test positive-class prevalence: 8.97%.
- No missing values were detected in either dataset.
- All input features were numeric.
- The held-out test dataset remains reserved for final evaluation.



In [4]:

import pandas as pd
import numpy as np

TARGET = "readmit_30"

train_df = pd.read_csv("/content/train_clean.csv")
test_df = pd.read_csv("/content/test_clean.csv")

X_train_full = train_df.drop(columns=[TARGET])
y_train_full = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of features:", X_train_full.shape[1])
print("Matching feature columns:",
      X_train_full.columns.equals(X_test.columns))
print("Training positive rate:",
      round(y_train_full.mean(), 4))
print("Test positive rate:",
      round(y_test.mean(), 4))
print("Missing values (train):",
      int(train_df.isna().sum().sum()))
print("Missing values (test):",
      int(test_df.isna().sum().sum()))
print("Non-numeric features:",
      X_train_full.select_dtypes(
          exclude="number"
      ).columns.tolist())


Training shape: (55976, 134)
Test shape: (13994, 134)
Number of features: 133
Matching feature columns: True
Training positive rate: 0.0897
Test positive rate: 0.0897
Missing values (train): 0
Missing values (test): 0
Non-numeric features: []



## 5. Training and Validation Split

### Decisions
- Reproduce the existing project's stratified training-validation split.
- Use 80% of the cleaned training partition for model training.
- Reserve 20% for validation.
- Set random_state=42 for reproducibility.
- Preserve approximately 9% positive-class prevalence.
- Keep the held-out test set untouched.
- Fit all future feature transformations using training data only.


### Results
- Successfully reproduced the stratified training-validation split.
- Training samples: 44,780.
- Validation samples: 11,196.
- Held-out test samples: 13,994.
- Input features: 133.
- Training positive-class prevalence: 8.97%.
- Validation positive-class prevalence: 8.97%.
- Training-validation row overlap: 0.
- Random seed: 42.
- The held-out test set remains untouched.
- Future preprocessing transformations will be fitted exclusively on training data.



In [5]:

from sklearn.model_selection import train_test_split

SEED = 42

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Held-out test samples:", len(X_test))

print("\nTraining positive rate:",
      round(y_train.mean(), 4))
print("Validation positive rate:",
      round(y_val.mean(), 4))

print("\nTraining/validation overlap:",
      len(set(X_train.index) & set(X_val.index)))

print("Number of features:", X_train.shape[1])


Training samples: 44780
Validation samples: 11196
Held-out test samples: 13994

Training positive rate: 0.0897
Validation positive rate: 0.0897

Training/validation overlap: 0
Number of features: 133



## 6. Feature Scaling for Patient Similarity

### Decisions
- Standardize numerical features before constructing similarity-based edges.
- Fit StandardScaler exclusively on training data.
- Apply the fitted transformation to validation features.
- Preserve all 133 input features.
- Keep the held-out test set untouched.
- Use the same scaled features for graph construction and GNN node attributes.


### Results
- Successfully standardized all 133 input features.
- Training matrix shape: (44,780, 133).
- Validation matrix shape: (11,196, 133).
- Training feature means were approximately zero.
- Training feature standard deviations were approximately one for the first five inspected features.
- No missing or infinite values were detected.
- StandardScaler was fitted exclusively on training data.
- Validation data was transformed using the fitted training scaler.
- The held-out test set remains untouched.



In [6]:

import numpy as np
from sklearn.preprocessing import StandardScaler

# Fit scaling parameters on training features only
gnn_scaler = StandardScaler()

X_train_scaled = gnn_scaler.fit_transform(
    X_train
).astype(np.float32)

X_val_scaled = gnn_scaler.transform(
    X_val
).astype(np.float32)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

print("\nTraining mean (first 5 features):")
print(np.round(X_train_scaled.mean(axis=0)[:5], 4))

print("\nTraining std (first 5 features):")
print(np.round(X_train_scaled.std(axis=0)[:5], 4))

print("\nMissing or infinite values (train):",
      int((~np.isfinite(X_train_scaled)).sum()))

print("Missing or infinite values (validation):",
      int((~np.isfinite(X_val_scaled)).sum()))

print("\nScaling completed successfully!")


Training shape: (44780, 133)
Validation shape: (11196, 133)

Training mean (first 5 features):
[ 0.  0.  0. -0. -0.]

Training std (first 5 features):
[1. 1. 1. 1. 1.]

Missing or infinite values (train): 0
Missing or infinite values (validation): 0

Scaling completed successfully!



## 7. Training Patient-Similarity Graph Construction

### Decisions
- Represent each training patient as a graph node.
- Build a k-nearest-neighbor graph using standardized clinical features.
- Connect each patient to five nearest neighbors.
- Use Euclidean distance as the initial similarity measure.
- Exclude each patient's self-neighbor.
- Create bidirectional edges for GraphSAGE message passing.
- Construct the graph exclusively from training features.
- Do not use readmission labels to determine graph edges.
- Keep validation and held-out test patients outside the training graph.


### Results
- Successfully constructed a k-nearest-neighbor patient-similarity graph.
- Number of patient nodes: 44,780.
- Number of directed edge entries: 357,670.
- Number of unique undirected patient connections: 178,835.
- Number of input features per node: 133.
- Number of positive training labels: 4,018.
- Initial kNN parameter: k = 5.
- Similarity metric: Euclidean distance on standardized features.
- The graph was converted to an undirected representation.
- No self-loops were detected.
- Graph construction used training features only.
- Readmission labels were not used to construct edges.
- Validation and held-out test patients were excluded from the training graph.
- The graph is ready for GraphSAGE model development.



In [7]:

import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

SEED = 42
K = 5

# Find nearest neighbors using training features only
knn = NearestNeighbors(
    n_neighbors=K + 1,
    metric="euclidean",
    algorithm="auto",
    n_jobs=-1
)

knn.fit(X_train_scaled)

distances, neighbors = knn.kneighbors(
    X_train_scaled,
    return_distance=True
)

# Remove the self-neighbor explicitly
source_nodes = []
target_nodes = []

for i in range(len(X_train_scaled)):
    valid_neighbors = [
        int(j) for j in neighbors[i]
        if int(j) != i
    ][:K]

    for j in valid_neighbors:
        source_nodes.append(i)
        target_nodes.append(j)

edge_index = torch.tensor(
    [source_nodes, target_nodes],
    dtype=torch.long
)

# Convert directed kNN edges to undirected edges
edge_index = to_undirected(
    edge_index,
    num_nodes=len(X_train_scaled)
)

# Create PyTorch Geometric training graph
train_graph = Data(
    x=torch.tensor(
        X_train_scaled,
        dtype=torch.float32
    ),
    edge_index=edge_index,
    y=torch.tensor(
        y_train.to_numpy(),
        dtype=torch.float32
    )
)

print("Number of nodes:", train_graph.num_nodes)
print("Number of edges:", train_graph.num_edges)
print("Number of features:", train_graph.num_node_features)
print("Graph is undirected:", train_graph.is_undirected())
print("Self-loops:", train_graph.has_self_loops())
print("Positive labels:", int(train_graph.y.sum().item()))
print("Training graph construction completed!")


Number of nodes: 44780
Number of edges: 357670
Number of features: 133
Graph is undirected: True
Self-loops: False
Positive labels: 4018
Training graph construction completed!



## 8. Graph Integrity and Connectivity Analysis

### Decisions
- Verify node and edge indices are valid.
- Confirm the graph contains no self-loops.
- Measure node degree distribution.
- Identify isolated patient nodes.
- Check that each node has at least five neighbors after graph symmetrization.
- Verify edge construction did not incorporate validation or test patients.
- Assess graph connectivity before training GraphSAGE.


### Results
- Successfully validated the training patient-similarity graph.
- Number of nodes: 44,780.
- Number of directed edge entries: 357,670.
- Minimum node index: 0.
- Maximum node index: 44,779.
- Minimum node degree: 5.
- Maximum node degree: 53.
- Mean node degree: 7.99.
- Median node degree: 7.
- No isolated nodes were detected.
- No nodes had degree below 5.
- No self-loops were detected.
- The graph was confirmed to be undirected.
- All edge indices were within valid node boundaries.
- Graph integrity assertions passed.
- These checks confirm structural validity, not predictive usefulness.



In [8]:

import torch
import numpy as np

edge_index = train_graph.edge_index
num_nodes = train_graph.num_nodes

# Count connections per node
degree = torch.bincount(
    edge_index[0],
    minlength=num_nodes
)

print("Graph Integrity Check")
print("------------------------------")

print("Nodes:", num_nodes)
print("Edges:", train_graph.num_edges)
print("Minimum node index:", int(edge_index.min()))
print("Maximum node index:", int(edge_index.max()))

print("\nDegree Statistics:")
print("Minimum degree:", int(degree.min()))
print("Maximum degree:", int(degree.max()))
print("Mean degree:", round(degree.float().mean().item(), 2))
print("Median degree:", float(degree.median().item()))

print("\nIsolated nodes:",
      int((degree == 0).sum()))

print("Nodes with degree below 5:",
      int((degree < 5).sum()))

print("\nSelf-loops:", train_graph.has_self_loops())
print("Undirected:", train_graph.is_undirected())

assert edge_index.min() >= 0
assert edge_index.max() < num_nodes
assert not train_graph.has_self_loops()
assert train_graph.is_undirected()

print("\nGraph integrity checks passed!")


Graph Integrity Check
------------------------------
Nodes: 44780
Edges: 357670
Minimum node index: 0
Maximum node index: 44779

Degree Statistics:
Minimum degree: 5
Maximum degree: 53
Mean degree: 7.99
Median degree: 7.0

Isolated nodes: 0
Nodes with degree below 5: 0

Self-loops: False
Undirected: True

Graph integrity checks passed!



## 9. GraphSAGE Model Architecture

### Decisions
- Use a two-layer GraphSAGE architecture for binary node classification.
- Input dimensionality: 133 patient features.
- Hidden dimensionality: 64.
- Output dimensionality: 1 logit per patient.
- Use ReLU activation between graph convolution layers.
- Apply dropout (0.3) during training.
- Return raw logits for use with BCEWithLogitsLoss.
- Initialize the model on the CUDA GPU.
- Verify the forward-pass output before training.


### Results
- Successfully implemented a two-layer GraphSAGE neural network.
- Input dimensionality: 133 features.
- Hidden dimensionality: 64.
- Output dimensionality: 1 logit per patient.
- Aggregation method: Mean.
- Activation: ReLU.
- Dropout rate: 0.3.
- Total trainable parameters: 17,217.
- Successfully moved the model and training graph to CUDA.
- Forward-pass output shape: (44,780,).
- No NaN outputs were detected.
- The model is ready for training.
- No model training has been performed yet.



In [9]:

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import SAGEConv

class ReadmissionGraphSAGE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x.squeeze(-1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gnn_model = ReadmissionGraphSAGE(
    input_dim=train_graph.num_node_features,
    hidden_dim=64,
    dropout=0.3
).to(device)

# Move the training graph to GPU
train_graph = train_graph.to(device)

# Test a forward pass without training
gnn_model.eval()

with torch.no_grad():
    sample_logits = gnn_model(
        train_graph.x,
        train_graph.edge_index
    )

print("Model:")
print(gnn_model)

print("\nTotal parameters:",
      sum(p.numel() for p in gnn_model.parameters()))

print("Output shape:", tuple(sample_logits.shape))
print("Output device:", sample_logits.device)
print("NaN outputs:",
      int(torch.isnan(sample_logits).sum()))

print("\nGraphSAGE forward pass successful!")


Model:
ReadmissionGraphSAGE(
  (conv1): SAGEConv(133, 64, aggr=mean)
  (conv2): SAGEConv(64, 1, aggr=mean)
)

Total parameters: 17217
Output shape: (44780,)
Output device: cuda:0
NaN outputs: 0

GraphSAGE forward pass successful!



## 10. Training Configuration and Class Imbalance Handling

### Decisions
- Use BCEWithLogitsLoss for binary node classification.
- Compute positive-class weighting from training labels only.
- Use Adam optimizer with learning rate 0.001.
- Apply weight decay of 0.0001.
- Use the full training graph for initial experiments.
- Monitor training loss before implementing validation and early stopping.
- Keep validation and test labels outside the training graph.


### Results
- Positive training samples: 4,018.
- Negative training samples: 40,762.
- Positive-class weight: 10.1448.
- Loss function: BCEWithLogitsLoss.
- Class weighting was calculated from training labels only.
- Optimizer: Adam.
- Learning rate: 0.001.
- Weight decay: 0.0001.
- The full training graph will be used for the initial experiment.
- Validation and held-out test labels remain excluded from training.
- Training has not yet started.



In [10]:

import torch
from torch import nn

# Calculate class imbalance from training labels
positive_count = train_graph.y.sum().item()
negative_count = train_graph.num_nodes - positive_count

pos_weight_value = negative_count / positive_count

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

# Binary classification loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Optimizer
optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

print("Positive samples:", int(positive_count))
print("Negative samples:", int(negative_count))
print("Positive class weight:", round(pos_weight_value, 4))

print("\nLoss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])

print("\nGraphSAGE training configuration completed!")


Positive samples: 4018
Negative samples: 40762
Positive class weight: 10.1448

Loss function: BCEWithLogitsLoss()
Optimizer: Adam
Learning rate: 0.001
Weight decay: 0.0001

GraphSAGE training configuration completed!



## 11. Initial GraphSAGE Training

### Decisions
- Train the two-layer GraphSAGE model for 20 epochs.
- Use the training-only patient-similarity graph.
- Apply class-weighted binary cross-entropy loss.
- Optimize model parameters using Adam.
- Record training loss and training PR-AUC.
- Confirm the model learns without numerical instability.
- Reserve validation-based model selection for a subsequent experiment.
- Keep the held-out test set untouched.


### Results
- Successfully trained the two-layer GraphSAGE model for 20 epochs.
- Training used the 44,780-node patient-similarity graph.
- Epoch 1 loss: 1.2797.
- Epoch 20 loss: 1.1991.
- Epoch 1 training PR-AUC: 0.1044.
- Epoch 20 training PR-AUC: 0.1632.
- Training loss decreased throughout the recorded checkpoints.
- Training PR-AUC increased throughout the recorded checkpoints.
- The model demonstrated learning on the training graph.
- No validation-based early stopping was applied.
- Predictive generalization has not yet been established.
- Training results must not be directly compared with held-out test metrics.
- The held-out test dataset remains untouched.



In [11]:

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

EPOCHS = 20
training_history = []

for epoch in range(1, EPOCHS + 1):

    # Training
    gnn_model.train()
    optimizer.zero_grad()

    logits = gnn_model(
        train_graph.x,
        train_graph.edge_index
    )

    loss = criterion(
        logits,
        train_graph.y
    )

    loss.backward()
    optimizer.step()

    # Training metrics
    gnn_model.eval()

    with torch.no_grad():
        train_logits = gnn_model(
            train_graph.x,
            train_graph.edge_index
        )

        train_probs = torch.sigmoid(
            train_logits
        ).cpu().numpy()

    train_labels = train_graph.y.cpu().numpy()

    train_prauc = average_precision_score(
        train_labels,
        train_probs
    )

    training_history.append({
        "Epoch": epoch,
        "Loss": loss.item(),
        "Train_PR_AUC": train_prauc
    })

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Loss: {loss.item():.4f} | "
            f"Train PR-AUC: {train_prauc:.4f}"
        )

training_df = pd.DataFrame(training_history)

print("\nTraining completed!")
print("Final loss:", round(training_df["Loss"].iloc[-1], 4))
print(
    "Final training PR-AUC:",
    round(training_df["Train_PR_AUC"].iloc[-1], 4)
)


Epoch 01/20 | Loss: 1.2797 | Train PR-AUC: 0.1044
Epoch 05/20 | Loss: 1.2450 | Train PR-AUC: 0.1335
Epoch 10/20 | Loss: 1.2226 | Train PR-AUC: 0.1463
Epoch 15/20 | Loss: 1.2089 | Train PR-AUC: 0.1550
Epoch 20/20 | Loss: 1.1991 | Train PR-AUC: 0.1632

Training completed!
Final loss: 1.1991
Final training PR-AUC: 0.1632



## 12. Inductive Validation Graph Construction

### Decisions
- Evaluate GraphSAGE on previously unseen validation patients.
- Connect each validation patient to five nearest training patients.
- Use the existing training-fitted StandardScaler.
- Reuse the training graph and its learned node representations.
- Do not use validation labels to construct graph connections.
- Prevent validation nodes from influencing training-node representations.
- Keep model parameters fixed during validation.
- Preserve the held-out test set for final evaluation.


### Results
- Successfully constructed an inductive validation graph.
- Training nodes: 44,780.
- Validation nodes: 11,196.
- Total nodes: 55,976.
- Original training edge entries: 357,670.
- Incoming training-to-validation edges: 55,980.
- Combined edge entries: 413,650.
- Each validation patient receives five incoming edges from training patients.
- No validation-to-training edges were introduced.
- Validation neighbors were selected using the training-fitted kNN index.
- Readmission labels were not used during edge construction.
- Model parameters remain unchanged.
- The held-out test set remains untouched.



In [12]:

import torch
import numpy as np
from torch_geometric.data import Data

K = 5
n_train = len(X_train_scaled)
n_val = len(X_val_scaled)

# Reuse training-fitted nearest-neighbor index
# Note: We query validation patients against training patients.
val_distances, val_neighbors = knn.kneighbors(
    X_val_scaled,
    n_neighbors=K,
    return_distance=True
)

# Build directed edges: training neighbor -> validation patient.
# This allows validation patients to aggregate training information
# without sending validation information back to training nodes.
val_sources = val_neighbors.reshape(-1)

val_targets = np.repeat(
    np.arange(n_val) + n_train,
    K
)

validation_edges = torch.tensor(
    np.vstack([val_sources, val_targets]),
    dtype=torch.long
)

# Combine training graph edges with incoming validation edges
combined_edge_index = torch.cat(
    [
        train_graph.edge_index.cpu(),
        validation_edges
    ],
    dim=1
)

# Combine training and validation features
combined_x = torch.tensor(
    np.vstack([X_train_scaled, X_val_scaled]),
    dtype=torch.float32
)

# Labels are NOT included in the inference graph.
validation_graph = Data(
    x=combined_x,
    edge_index=combined_edge_index
).to(device)

print("Training nodes:", n_train)
print("Validation nodes:", n_val)
print("Combined nodes:", validation_graph.num_nodes)
print("Combined edge entries:", validation_graph.num_edges)
print("Validation incoming edges:", validation_edges.shape[1])
print("Validation-to-training edges:",
      int((validation_edges[1] < n_train).sum()))
print("Validation graph prepared successfully!")


Training nodes: 44780
Validation nodes: 11196
Combined nodes: 55976
Combined edge entries: 413650
Validation incoming edges: 55980
Validation-to-training edges: 0
Validation graph prepared successfully!



## 13. GraphSAGE Validation Evaluation

### Decisions
- Evaluate the trained GraphSAGE model on unseen validation patients.
- Use the inductive validation graph with directed incoming edges.
- Keep model parameters fixed during inference.
- Compute PR-AUC and ROC-AUC from prediction probabilities.
- Compute precision, recall, and F1-score at threshold 0.5.
- Inspect the confusion matrix and predicted positive counts.
- Do not use the held-out test set.
- Treat results as an initial validation assessment.


### Results
- Successfully evaluated GraphSAGE on 11,196 unseen validation patients.
- Positive validation cases: 1,004.
- PR-AUC (Average Precision): 0.1493.
- ROC-AUC: 0.6336.
- Precision at threshold 0.5: 0.1293.
- Recall at threshold 0.5: 0.5876.
- F1-score at threshold 0.5: 0.2120.
- True positives: 590.
- False positives: 3,972.
- False negatives: 414.
- True negatives: 6,220.
- Predicted positive cases: 4,562.
- Prediction probability range: 0.0245 to 0.9331.
- The model identified 58.76% of actual readmissions but produced many false positives.
- Validation PR-AUC exceeded the positive-class prevalence baseline of approximately 0.0897.
- The results demonstrate modest predictive discrimination under the current graph construction strategy.
- No validation-based threshold optimization or model selection has been performed yet.
- The held-out test dataset remains untouched.


In [13]:

import torch
import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Evaluate the trained model
gnn_model.eval()

with torch.inference_mode():
    all_logits = gnn_model(
        validation_graph.x,
        validation_graph.edge_index
    )

    # Validation nodes follow the training nodes
    val_logits = all_logits[n_train:]

    val_probs = torch.sigmoid(
        val_logits
    ).cpu().numpy()

# Ground-truth labels used only for evaluation
val_true = y_val.to_numpy()

# Preliminary threshold
THRESHOLD = 0.5
val_pred = (val_probs >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    val_true,
    val_pred,
    labels=[0, 1]
).ravel()

print("GraphSAGE Validation Results")
print("------------------------------")
print("PR-AUC:",
      round(average_precision_score(val_true, val_probs), 4))
print("ROC-AUC:",
      round(roc_auc_score(val_true, val_probs), 4))
print("Precision:",
      round(precision_score(val_true, val_pred, zero_division=0), 4))
print("Recall:",
      round(recall_score(val_true, val_pred, zero_division=0), 4))
print("F1:",
      round(f1_score(val_true, val_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)

print("\nValidation patients:", len(val_true))
print("Actual positives:", int(val_true.sum()))
print("Predicted positives:", int(val_pred.sum()))

print("\nProbability range:",
      round(float(val_probs.min()), 4),
      "to",
      round(float(val_probs.max()), 4))


GraphSAGE Validation Results
------------------------------
PR-AUC: 0.1493
ROC-AUC: 0.6336
Precision: 0.1293
Recall: 0.5876
F1: 0.212

Confusion Matrix:
TP: 590 FP: 3972 FN: 414 TN: 6220

Validation patients: 11196
Actual positives: 1004
Predicted positives: 4562

Probability range: 0.0245 to 0.9331



## 14. GraphSAGE Threshold Optimization

### Decisions
- Optimize the classification threshold using validation predictions.
- Evaluate thresholds from 0.05 to 0.95.
- Select the threshold maximizing validation F1-score.
- Report precision and recall at the selected threshold.
- Avoid using held-out test labels during optimization.
- Treat the selected threshold as provisional until final model selection is complete.


### Results
- Evaluated classification thresholds from 0.05 to 0.95.
- Selected the threshold maximizing validation F1-score.
- Best validation threshold: 0.53.
- Precision at the selected threshold: 0.1385.
- Recall at the selected threshold: 0.4890.
- F1-score at the selected threshold: 0.2159.
- At the default threshold of 0.50, F1-score was 0.2120.
- Threshold optimization produced a small improvement in F1-score.
- Increasing the threshold improved precision but reduced recall.
- Validation PR-AUC remained 0.1493 because it is threshold-independent.
- The selected threshold was determined exclusively from validation predictions.
- The held-out test dataset remains untouched.



In [14]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(0.05, 0.951, 0.01):

    predictions = (
        val_probs >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            val_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            val_true,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            val_true,
            predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

BEST_THRESHOLD = float(best_row["Threshold"])

print("Best Validation Threshold:", BEST_THRESHOLD)
print("Precision:", round(best_row["Precision"], 4))
print("Recall:", round(best_row["Recall"], 4))
print("F1:", round(best_row["F1"], 4))

print("\nTop 10 thresholds:")
display(
    threshold_df.sort_values(
        "F1", ascending=False
    ).head(10).round(4)
)


Best Validation Threshold: 0.53
Precision: 0.1385
Recall: 0.489
F1: 0.2159

Top 10 thresholds:


,Threshold,Precision,Recall,F1
48,0.53,0.1385,0.4890,0.2159
49,0.54,0.1400,0.4522,0.2137
47,0.52,0.1343,0.5199,0.2135
46,0.51,0.1320,0.5558,0.2133
45,0.50,0.1293,0.5876,0.2120
50,0.55,0.1413,0.4114,0.2104
44,0.49,0.1245,0.6145,0.2071
51,0.56,0.1429,0.3725,0.2065
52,0.57,0.1452,0.3376,0.2031
43,0.48,0.1207,0.6365,0.2029



## 15. GraphSAGE Model Checkpoint

### Decisions
- Save the trained GraphSAGE model parameters.
- Record the architecture configuration.
- Preserve the validation-selected threshold.
- Save training history and validation metrics.
- Avoid saving patient-level training or test data in the checkpoint.
- Preserve the current model before further experimentation.


### Results
- Successfully saved the trained GraphSAGE model checkpoint.
- Checkpoint path: results/graphsage_checkpoint.pt.
- Checkpoint file size: 0.069 MB.
- Model architecture: Two-layer GraphSAGE (133 → 64 → 1).
- Training epochs: 20.
- Selected validation threshold: 0.53.
- Validation PR-AUC: 0.1493.
- Validation ROC-AUC: 0.6336.
- Validation F1-score: 0.2159.
- The checkpoint stores model parameters and configuration.
- Patient-level data was not included in the checkpoint.
- The training-fitted scaler and kNN reference are not included.
- The held-out test set remains untouched.



In [15]:

import torch
import os

os.makedirs("results", exist_ok=True)

checkpoint = {
    "model_state_dict": gnn_model.state_dict(),
    "input_dim": 133,
    "hidden_dim": 64,
    "dropout": 0.3,
    "epochs": 20,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "k_neighbors": 5,
    "best_threshold": BEST_THRESHOLD,
    "validation_pr_auc": 0.1493,
    "validation_roc_auc": 0.6336,
    "validation_f1": 0.2159,
    "random_seed": 42
}

checkpoint_path = "results/graphsage_checkpoint.pt"

torch.save(checkpoint, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)
print("Selected threshold:", checkpoint["best_threshold"])
print("Validation PR-AUC:", checkpoint["validation_pr_auc"])

print(
    "Checkpoint size (MB):",
    round(os.path.getsize(checkpoint_path) / 1024**2, 3)
)


Checkpoint saved: results/graphsage_checkpoint.pt
Selected threshold: 0.53
Validation PR-AUC: 0.1493
Checkpoint size (MB): 0.069



## 16. Validation-Based Training Configuration

### Decisions
- Initialize a new GraphSAGE model from scratch.
- Preserve the original 20-epoch model checkpoint.
- Use the same two-layer architecture and training graph.
- Monitor validation PR-AUC for model selection.
- Use early stopping with patience of 10 epochs.
- Limit training to a maximum of 100 epochs.
- Restore the model weights with the highest validation PR-AUC.
- Optimize the classification threshold after selecting model weights.
- Keep the held-out test dataset untouched.




In [16]:

import torch
from torch import nn
import numpy as np
import random

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# New model (do not overwrite original)
gnn_model_es = ReadmissionGraphSAGE(
    input_dim=133,
    hidden_dim=64,
    dropout=0.3
).to(device)

# Same class-weighted loss
criterion_es = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# New optimizer
optimizer_es = torch.optim.Adam(
    gnn_model_es.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

MAX_EPOCHS = 100
PATIENCE = 10

print("New GraphSAGE model initialized.")
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Model parameters:",
      sum(p.numel() for p in gnn_model_es.parameters()))
print("Validation metric: PR-AUC")
print("Original model preserved:", gnn_model is not gnn_model_es)


New GraphSAGE model initialized.
Maximum epochs: 100
Early stopping patience: 10
Model parameters: 17217
Validation metric: PR-AUC
Original model preserved: True



## 17. GraphSAGE Training with Early Stopping

### Decisions
- Train the newly initialized GraphSAGE model.
- Use the training-only patient-similarity graph for gradient updates.
- Monitor validation PR-AUC after each epoch.
- Use the inductive validation graph for evaluation.
- Apply early stopping with patience of 10 epochs.
- Train for a maximum of 100 epochs.
- Save and restore the model state with the highest validation PR-AUC.
- Do not optimize the classification threshold during training.
- Keep the held-out test dataset untouched.


### Results
- Successfully trained a new GraphSAGE model using validation-based early stopping.
- Maximum allowed epochs: 100.
- Early stopping patience: 10 epochs.
- Actual training duration: 53 epochs.
- Best model epoch: 43.
- Best validation PR-AUC: 0.1456.
- The best-performing model weights were successfully restored.
- The original 20-epoch GraphSAGE achieved validation PR-AUC of 0.1493.
- The early-stopping experiment did not outperform the original model.
- Both models used the same architecture and validation graph, but different random initializations.
- The results do not establish that early stopping reduced performance.
- The held-out test set remains untouched.



In [17]:

import torch
import numpy as np
import pandas as pd
import copy

from sklearn.metrics import average_precision_score

best_val_prauc = -1.0
best_epoch = 0
best_model_state = None
patience_counter = 0

es_history = []

# Validation labels are used for model selection only
val_labels = y_val.to_numpy()

for epoch in range(1, MAX_EPOCHS + 1):

    # -------------------------------
    # Training on training graph only
    # -------------------------------
    gnn_model_es.train()
    optimizer_es.zero_grad()

    train_logits = gnn_model_es(
        train_graph.x,
        train_graph.edge_index
    )

    loss = criterion_es(
        train_logits,
        train_graph.y
    )

    loss.backward()
    optimizer_es.step()

    # -------------------------------
    # Validation: no gradient updates
    # -------------------------------
    gnn_model_es.eval()

    with torch.inference_mode():
        all_logits = gnn_model_es(
            validation_graph.x,
            validation_graph.edge_index
        )

        validation_logits = all_logits[n_train:]

        validation_probs = torch.sigmoid(
            validation_logits
        ).cpu().numpy()

    val_prauc = average_precision_score(
        val_labels,
        validation_probs
    )

    es_history.append({
        "Epoch": epoch,
        "Train_Loss": float(loss.item()),
        "Val_PR_AUC": float(val_prauc)
    })

    # -------------------------------
    # Early stopping
    # -------------------------------
    if val_prauc > best_val_prauc + 1e-6:

        best_val_prauc = val_prauc
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            gnn_model_es.state_dict()
        )

        patience_counter = 0

    else:
        patience_counter += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Val PR-AUC: {val_prauc:.4f} | "
            f"Best: {best_val_prauc:.4f}"
        )

    if patience_counter >= PATIENCE:
        print(
            f"\nEarly stopping triggered "
            f"at epoch {epoch}."
        )
        break

# Restore the best model
assert best_model_state is not None

gnn_model_es.load_state_dict(best_model_state)
gnn_model_es.eval()

es_history_df = pd.DataFrame(es_history)

print("\nTraining completed!")
print("Total epochs:", len(es_history_df))
print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", round(best_val_prauc, 4))
print("Best model weights restored successfully!")


Epoch 001 | Loss: 1.3565 | Val PR-AUC: 0.0881 | Best: 0.0881
Epoch 005 | Loss: 1.2680 | Val PR-AUC: 0.1084 | Best: 0.1084
Epoch 010 | Loss: 1.2305 | Val PR-AUC: 0.1262 | Best: 0.1262
Epoch 015 | Loss: 1.2172 | Val PR-AUC: 0.1366 | Best: 0.1366
Epoch 020 | Loss: 1.2048 | Val PR-AUC: 0.1407 | Best: 0.1407
Epoch 025 | Loss: 1.1935 | Val PR-AUC: 0.1426 | Best: 0.1426
Epoch 030 | Loss: 1.1885 | Val PR-AUC: 0.1433 | Best: 0.1437
Epoch 035 | Loss: 1.1797 | Val PR-AUC: 0.1444 | Best: 0.1444
Epoch 040 | Loss: 1.1761 | Val PR-AUC: 0.1453 | Best: 0.1453
Epoch 045 | Loss: 1.1727 | Val PR-AUC: 0.1456 | Best: 0.1456
Epoch 050 | Loss: 1.1676 | Val PR-AUC: 0.1455 | Best: 0.1456

Early stopping triggered at epoch 53.

Training completed!
Total epochs: 53
Best epoch: 43
Best validation PR-AUC: 0.1456
Best model weights restored successfully!



## 18. GraphSAGE Model Selection

### Decisions
- Compare both GraphSAGE experiments on the same validation records.
- Evaluate PR-AUC and ROC-AUC.
- Optimize the classification threshold for the early-stopping model using validation data.
- Compare validation Precision, Recall, and F1-score.
- Select the preferred model based primarily on validation PR-AUC.
- Keep the held-out test set untouched.


### Results
- Compared two GraphSAGE models on the same 11,196 validation patients.
- Initial GraphSAGE (20 epochs):
  - PR-AUC: 0.1493.
  - ROC-AUC: 0.6336.
  - Selected threshold: 0.53.
  - Precision: 0.1385.
  - Recall: 0.4890.
  - F1-score: 0.2159.
- Early-Stopping GraphSAGE:
  - PR-AUC: 0.1456.
  - ROC-AUC: 0.6308.
  - Selected threshold: 0.53.
  - Precision: 0.1354.
  - Recall: 0.4821.
  - F1-score: 0.2114.
- The initial GraphSAGE achieved slightly higher scores across all reported metrics.
- Selected the initial 20-epoch GraphSAGE model based on validation PR-AUC.
- The performance difference was small and was not tested for statistical significance.
- The selected classification threshold is 0.53.
- The held-out test dataset remains untouched.



In [18]:

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Evaluate the early-stopping model
gnn_model_es.eval()

with torch.inference_mode():
    es_logits = gnn_model_es(
        validation_graph.x,
        validation_graph.edge_index
    )[n_train:]

    es_probs = torch.sigmoid(
        es_logits
    ).cpu().numpy()

# Optimize threshold on validation data
es_thresholds = np.arange(0.05, 0.951, 0.01)

es_f1_scores = [
    f1_score(
        val_true,
        (es_probs >= t).astype(int),
        zero_division=0
    )
    for t in es_thresholds
]

best_es_threshold = float(
    es_thresholds[np.argmax(es_f1_scores)]
)

# Compare the two models
models_to_compare = [
    ("Initial GraphSAGE", val_probs, BEST_THRESHOLD),
    ("Early-Stopping GraphSAGE", es_probs, best_es_threshold)
]

comparison_rows = []

for name, probabilities, threshold in models_to_compare:

    predictions = (
        probabilities >= threshold
    ).astype(int)

    comparison_rows.append({
        "Model": name,
        "PR-AUC": average_precision_score(
            val_true, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            val_true, probabilities
        ),
        "Threshold": threshold,
        "Precision": precision_score(
            val_true, predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            val_true, predictions,
            zero_division=0
        ),
        "F1": f1_score(
            val_true, predictions,
            zero_division=0
        )
    })

gnn_comparison_df = pd.DataFrame(comparison_rows)

display(gnn_comparison_df.round(4))


,Model,PR-AUC,ROC-AUC,Threshold,Precision,Recall,F1
0,Initial GraphSAGE,0.1493,0.6336,0.53,0.1385,0.4890,0.2159
1,Early-Stopping GraphSAGE,0.1456,0.6308,0.53,0.1354,0.4821,0.2114



## 19. Final Test Graph Construction

### Decisions
- Use the selected initial GraphSAGE model.
- Preserve the original training graph and training reference patients.
- Transform test features using the training-fitted StandardScaler.
- Connect each test patient to five nearest training patients.
- Use directed training-to-test edges for inductive inference.
- Do not use test labels to construct graph connections.
- Keep model parameters and classification threshold fixed.
- Prepare the graph for one final held-out test evaluation.


### Results
- Successfully constructed the final inductive test graph.
- Training reference nodes: 44,780.
- Held-out test nodes: 13,994.
- Total graph nodes: 58,774.
- Incoming training-to-test edges: 69,970.
- Total directed edge entries: 427,640.
- Each test patient was connected to five nearest training patients.
- Test features were standardized using the training-fitted StandardScaler.
- Neighbors were selected using the training-fitted kNN index.
- No test labels were used during graph construction.
- The selected GraphSAGE model and threshold remain fixed.
- The graph is ready for final held-out test evaluation.



In [19]:

import numpy as np
import torch
from torch_geometric.data import Data

# Transform test features using training-fitted scaler
X_test_scaled = gnn_scaler.transform(
    X_test
).astype(np.float32)

n_test = len(X_test_scaled)

# Find five training neighbors for each test patient
test_distances, test_neighbors = knn.kneighbors(
    X_test_scaled,
    n_neighbors=5,
    return_distance=True
)

# Directed edges: training patient -> test patient
test_sources = test_neighbors.reshape(-1)

test_targets = np.repeat(
    np.arange(n_test) + n_train,
    5
)

test_edges = torch.tensor(
    np.vstack([test_sources, test_targets]),
    dtype=torch.long
)

# Retain the original training graph
test_combined_edges = torch.cat(
    [
        train_graph.edge_index.cpu(),
        test_edges
    ],
    dim=1
)

test_combined_x = torch.tensor(
    np.vstack([X_train_scaled, X_test_scaled]),
    dtype=torch.float32
)

# No labels are included in the test graph
test_graph = Data(
    x=test_combined_x,
    edge_index=test_combined_edges
).to(device)

print("Training nodes:", n_train)
print("Test nodes:", n_test)
print("Combined nodes:", test_graph.num_nodes)
print("Incoming test edges:", test_edges.shape[1])
print("Total edge entries:", test_graph.num_edges)
print("Test graph created successfully!")


Training nodes: 44780
Test nodes: 13994
Combined nodes: 58774
Incoming test edges: 69970
Total edge entries: 427640
Test graph created successfully!



## 20. Final GraphSAGE Held-Out Test Evaluation

### Decisions
- Use the initial 20-epoch GraphSAGE model selected on validation PR-AUC.
- Apply the fixed validation-selected threshold of 0.53.
- Evaluate all 13,994 held-out test patients.
- Use directed training-to-test edges for inductive inference.
- Compute PR-AUC, ROC-AUC, Precision, Recall, and F1-score.
- Record confusion matrix counts.
- Do not perform further tuning using held-out test outcomes.
- Compare results with the existing baseline models.


### Results
- Successfully evaluated the selected GraphSAGE model on 13,994 held-out test patients.
- Model: Two-layer GraphSAGE trained for 20 epochs.
- Graph construction: 5-nearest-neighbor patient similarity using standardized features.
- Inference strategy: Directed training-to-test edges.
- Fixed validation-selected threshold: 0.53.
- Test PR-AUC: 0.1423.
- Test ROC-AUC: 0.6114.
- Test Precision: 0.1287.
- Test Recall: 0.4375.
- Test F1-score: 0.1989.
- True positives: 549.
- False positives: 3,717.
- False negatives: 706.
- True negatives: 9,022.
- Actual positive test cases: 1,255.
- Predicted positive cases: 4,266.
- GraphSAGE did not outperform the existing tabular baselines in this experiment.
- These findings apply to the evaluated graph construction and model configuration only.
- No model tuning was performed using the held-out test outcomes.



In [20]:

import torch
import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Use the selected original GraphSAGE model
gnn_model.eval()

with torch.inference_mode():

    all_test_logits = gnn_model(
        test_graph.x,
        test_graph.edge_index
    )

    # Extract test patient outputs only
    test_logits = all_test_logits[n_train:]

    gnn_test_probs = torch.sigmoid(
        test_logits
    ).cpu().numpy()

# Held-out labels are used ONLY for evaluation
gnn_test_true = y_test.to_numpy()

# Fixed threshold selected on validation
FINAL_THRESHOLD = 0.53

gnn_test_pred = (
    gnn_test_probs >= FINAL_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    gnn_test_true,
    gnn_test_pred,
    labels=[0, 1]
).ravel()

print("Final GraphSAGE Test Results")
print("-----------------------------")

print("PR-AUC:",
      round(average_precision_score(
          gnn_test_true, gnn_test_probs
      ), 4))

print("ROC-AUC:",
      round(roc_auc_score(
          gnn_test_true, gnn_test_probs
      ), 4))

print("Precision:",
      round(precision_score(
          gnn_test_true, gnn_test_pred,
          zero_division=0
      ), 4))

print("Recall:",
      round(recall_score(
          gnn_test_true, gnn_test_pred,
          zero_division=0
      ), 4))

print("F1:",
      round(f1_score(
          gnn_test_true, gnn_test_pred,
          zero_division=0
      ), 4))

print("\nThreshold:", FINAL_THRESHOLD)
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
print("Test patients:", len(gnn_test_true))
print("Positive cases:", int(gnn_test_true.sum()))
print("Predicted positives:", int(gnn_test_pred.sum()))


Final GraphSAGE Test Results
-----------------------------
PR-AUC: 0.1423
ROC-AUC: 0.6114
Precision: 0.1287
Recall: 0.4375
F1: 0.1989

Threshold: 0.53
TP: 549 FP: 3717 FN: 706 TN: 9022
Test patients: 13994
Positive cases: 1255
Predicted positives: 4266
